<div align="center">
    <picture>
        <source media="(prefers-color-scheme: dark)" srcset="https://github.com/typedef-ai/fenic/raw/main/docs/images/typedef-fenic-logo-dark.png">
        <img src="https://github.com/typedef-ai/fenic/raw/main/docs/images/typedef-fenic-logo-dark.png" alt="fenic, by typedef" width="50%">
    </picture>
</div>

# Fenic On-Call Triage Agent (LangChain/LangGraph)

To run this notebook just press _"Run All"_ <span style="opacity:.8;">(in Google Colab: <b>Runtime ▸ Run all</b>)</span>

<p align="center">
  <a href="https://docs.fenic.ai">Read the Docs</a> •
  <a href="https://discord.com/invite/GdqF3J7huR">Join Discord</a> •
  <a href="https://github.com/typedef-ai/fenic">⭐️ Star fenic</a>
</p>

To install fenic locally, just follow the instructions on the [Github Repo](https://github.com/typedef-ai/fenic)


If this notebook helps, please give <a href="https://github.com/typedef-ai/fenic" target="_blank" rel="noopener noreferrer">fenic</a> a ⭐️ — it really helps!


## Problem

This notebook walks through a practical, end-to-end triage pipeline for production logs using Fenic for data processing and LangGraph/LangChain for orchestration.

The tutorial follows the flow: template-based parsing, service metadata enrichment via joins, severity assessment with business context, semantic clustering,MCP tools, and a LangGraph orchestration policy that allows you to ask questions of your error logs.

### **What you’ll build**

* **Parsing without regex:** Extract `timestamp / level / service / message` from mixed formats (ISO, syslog-style, Python logging) via small templates.

* **Enrichment:** Join logs with a lightweight service catalog (owner, on-call channel, tier, SLO targets, env, region).

* **Severity tagging:** Digit-safe rules (keeps `5xx`, error keywords, OperationalError/Timeout, nginx upstream issues) → `info | warn | error` \+ a numeric severity score.

* **Semantic clustering:** Embed enriched text and run **stratified K-Means** (per severity) to avoid mixing INFO with ERROR.

* **Deliverables:** CSV/JSON/Markdown artifacts in `/content/out`.

* **MCP:** Expose the Fenic tool and one info tool that return structured rows LangGraph can call.

* **LangGraph:** Call the MCP once and make a simple policy decision: `page-oncall` vs `log-only`.

### **Who this is for**

* SREs and platform engineers who want a reproducible triage loop in notebooks.

* Data/ML folks prototyping embeddings-driven incident grouping without wiring a whole platform.

## Install Required Libs and Packages

In [ ]:
# %%capture
# --- Install core packages (Colab-friendly) ---
!pip -q install --upgrade fenic
!pip -q install --upgrade langgraph

# Optional: minimal langchain core + OpenAI wiring for LangGraph appendix later
!pip -q install --upgrade langchain langchain-openai

# Installs for clustering (lightweight)
!pip -q install scikit-learn

# Installs for the MCP server section
!pip -q install "fenic[mcp]" uvicorn fastmcp websockets starlette

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.9/73.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.6/229.6 kB 9.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

## Add OpenAI Key to Use LLM Model and Embeddings for Classifying Logs

In [ ]:
import os
import getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

OpenAI API Key:··········


## **Step 1: Install & start a fresh Fenic session with embeddings (deterministic, tutorial-ready)**



**Goals**

* Create a **new Fenic session** every run so settings don’t leak between cells/demos.

* Configure two **model aliases** the rest of the notebook can rely on:

  * `mini` → inexpensive LLM for quick classification/summarization.

  * `embed` → vector model used by clustering.

* **Verify** embeddings work up front (fail fast), so downstream cells are simple and don’t need defensive fallbacks.

**Why this order**

1. Make output folders so artifacts have a predictable home.

2. Build a **SemanticConfig** with explicit aliases (`mini`, `embed`) and set them as defaults.

3. Use a **unique `app_name`** per run to force a clean `Session` (avoids “sticky” configs).

4. **Probe** `semantic.embed` with the alias to confirm API/key wiring now, not later.

**What if no API key?**

* This tutorial expects embeddings; if `OPENAI_API_KEY` is missing, we raise an assertion right away.

* (If you want a purely deterministic walkthrough, you can remove the assertion and skip embedding cells but the accepted Type/Fenic pattern uses semantic steps.)

**Common pitfalls this avoids**

* “No embedding model configured” at clustering time.

* Accidentally reusing a prior session with stale semantic settings.

* Silent failures from rate limits: we set **RPM/TPM** caps to conservative, demo-safe values.


In [ ]:
# --- Step 1: Set Fenic session and other configs ---

import os, time
from pathlib import Path
import fenic as fc

# Colab project folders
BASE = Path("/content")
(BASE / "data" / "logs").mkdir(parents=True, exist_ok=True)
(BASE / "out").mkdir(parents=True, exist_ok=True)

# --- Credentials ---
# In Colab: Runtime ▸ Run time env vars…  add OPENAI_API_KEY
OPENAI_KEY = os.environ.get("OPENAI_API_KEY")
assert OPENAI_KEY and len(OPENAI_KEY) > 10, "Set OPENAI_API_KEY in Runtime ▸ Runtime env vars."

# --- Session config ---
# app name avoids reusing stale state between runs
APP_NAME = f"fenic_demo_{int(time.time())}"

semantic_cfg = fc.SemanticConfig(
    language_models={
        # Small/fast LLM for classification/summarization later
        "mini": fc.OpenAILanguageModel(model_name="gpt-4o-mini", rpm=500, tpm=200_000),
    },
    embedding_models={
        # Embeddings used for semantic clustering
        "embed": fc.OpenAIEmbeddingModel(model_name="text-embedding-3-small", rpm=3000, tpm=3_000_000),
    },
    default_language_model="mini",
    default_embedding_model="embed",
)

session = fc.Session.get_or_create(fc.SessionConfig(app_name=APP_NAME, semantic=semantic_cfg))

print("✅ Fenic session ready")
print(" App:", APP_NAME)
print(" LMs:", list(semantic_cfg.language_models.keys()))
print(" EMBs:", list(semantic_cfg.embedding_models.keys()))
print(" Default LM :", semantic_cfg.default_language_model)
print(" Default EMB:", semantic_cfg.default_embedding_model)

✅ Fenic session ready
 App: fenic_demo_1759510274
 LMs: ['mini']
 EMBs: ['embed']
 Default LM : mini
 Default EMB: embed


## **Step 2: Ingest raw logs (stable multi-line capture → `logs_df`)**


**What this cell does**

* Creates a **tiny, in-memory dataset** of 10 log entries (2 “files”) and loads them into a **single Fenic DataFrame** with canonical columns:

  * `source_path`, `lineno`, `text`

* Keeps things **deterministic**: no file I/O, no globbing, no multi-line detection logic here—the sample rows already reflect the final framing (stack traces are a single `text` entry).

**Why this version**

* Matches Kostas’ guidance: pick one path and keep it simple.

* Makes the notebook **self-contained** and runnable end-to-end without external files.

* Defers all structure/semantics to later steps (templates, unnest, LLM, severity), so each cell has a single responsibility.

**Notes for readers**

* `lineno` counts **entries** within each logical source (the faux file path), not physical lines.

* You can swap the `rows` list with your own samples later without touching downstream cells.

* The quick `.count()` and `.show()` confirm the frame is loaded and shaped as expected.

In [ ]:
# --- Step 2: Ingest raw .log lines into a Fenic DataFrame ---

import fenic as fc

# Load raw log entries as rows: (source_path, lineno, text).

rows = [
    # app1.log (5 entries)
    ("/content/data/logs/app1.log", 1, "2025-09-20T10:14:02Z [INFO] web: GET /health 200 in 3ms"),
    ("/content/data/logs/app1.log", 2, "2025-09-20T10:14:05Z [WARN] db: Connection slow for shard=3 latency=412ms"),
    ("/content/data/logs/app1.log", 3, """2025-09-20T10:14:09Z [ERROR] api: TimeoutError on /v1/users after 30010ms trace_id=abc-123
Traceback (most recent call last):
  File "/srv/app/handlers/user.py", line 214, in handle
    resp = client.get(url, timeout=30)
  File "/usr/local/lib/python3.11/site-packages/requests/api.py", line 73, in get
    return request('get', url, params=params, **kwargs)
requests.exceptions.Timeout: The request timed out"""),
    ("/content/data/logs/app1.log", 4, "2025-09-20T10:14:12Z [INFO] worker: job=metrics-flush ok"),
    ("/content/data/logs/app1.log", 5, "2025-09-20T10:14:14Z [ERROR] db: UNIQUE constraint failed: users.email id=9843e5f2-778c-4f5f-a7f2-1b9c9d1a"),

    # edge_app.log (5 entries)
    ("/content/data/logs/edge_app.log", 1, "Sep 20 10:14:01 edge-1 nginx[1024]: 502 upstream prematurely closed connection while reading response header"),
    ("/content/data/logs/edge_app.log", 2, 'Sep 20 10:14:04 edge-1 nginx[1024]: 499 client closed connection while waiting for request, request: "GET /"'),
    ("/content/data/logs/edge_app.log", 3, "Sep 20 10:14:09 edge-1 nginx[1024]: 500 upstream sent too big header while reading response header from upstream"),
    ("/content/data/logs/edge_app.log", 4, """2025-09-20 10:14:11,771 - payment-api - ERROR - psycopg2.OperationalError: could not connect to server: Connection refused
Is the server running on host "db.internal" (10.0.0.12) and accepting
TCP/IP connections on port 5432?"""),
    ("/content/data/logs/edge_app.log", 5, "2025-09-20 10:14:13,120 - payment-api - WARN - retrying in 2s (attempt 2/5)"),
]

# 2) Build Fenic DataFrame
logs_df = session.create_dataframe({
    "source_path": [r[0] for r in rows],
    "lineno":      [r[1] for r in rows],
    "text":        [r[2] for r in rows],
})

# 3) Sanity peek
print("Columns:", logs_df.columns)
print("Total rows:", logs_df.count())
logs_df.limit(8).show()

Columns: ['source_path', 'lineno', 'text']
Total rows: 10
┌─────────────────────────────────┬────────┬───────────────────────────────────────────────────────┐
│ source_path                     ┆ lineno ┆ text                                                  │
╞═════════════════════════════════╪════════╪═══════════════════════════════════════════════════════╡
│ /content/data/logs/app1.log     ┆ 1      ┆ 2025-09-20T10:14:02Z [INFO] web: GET /health 200 in   │
│                                 ┆        ┆ 3ms                                                   │
│ /content/data/logs/app1.log     ┆ 2      ┆ 2025-09-20T10:14:05Z [WARN] db: Connection slow for   │
│                                 ┆        ┆ shard=3 latency=412ms                                 │
│ /content/data/logs/app1.log     ┆ 3      ┆ 2025-09-20T10:14:09Z [ERROR] api: TimeoutError on     │
│                                 ┆        ┆ /v1/users after 30010ms trace_id=abc-123              │
│                                

## **Step 3: Template parsing via `text.extract` \+ `unnest` → `parsed_df`**

**What this cell does**

* Parses each raw `text` into structured fields using **templates** (no regex):

  * Core: `timestamp`, `level`, `service`, `message`

  * Optional: `trace_id`

* Supports **three formats in one pass** by extracting 4 struct columns:

  * `p1` (ISO app logs), `p2` (syslog-ish), `p3` (Python logging), `tid` (trace id).

* Flattens those structs with **`DataFrame.unnest(...)`** and then **coalesces** fields to a single canonical schema.

In [ ]:
# --- Step 3: Parse via templates + unnest ---

TEMPLATE_ISO    = "${timestamp:none} [${level:none}] ${service:none}: ${message:none}"
TEMPLATE_SYSLOG = "${timestamp:none} ${host:none} ${service:none}[${pid:none}]: ${message:none}"
TEMPLATE_PYLOG  = "${timestamp:none} - ${service:none} - ${level:none} - ${message:none}"

def parse_lines(df):
    # Extract candidate structs
    base = df.select(
        "source_path", "lineno", "text",
        fc.text.extract(fc.col("text"), TEMPLATE_ISO).alias("p1"),
        fc.text.extract(fc.col("text"), TEMPLATE_SYSLOG).alias("p2"),
        fc.text.extract(fc.col("text"), TEMPLATE_PYLOG).alias("p3"),
        fc.text.extract(fc.col("text"), "trace_id=${trace_id:none}").alias("tid"),
    )

    # Unnest p1 -> alias to ts1/lvl1/svc1/msg1 (avoid name clashes by aliasing immediately)
    s1 = (
        base
        .unnest("p1")
        .select(
            "source_path", "lineno", "text", "p2", "p3", "tid",
            fc.col("timestamp").alias("ts1"),
            fc.col("level").alias("lvl1"),
            fc.col("service").alias("svc1"),
            fc.col("message").alias("msg1"),
        )
    )

    # Unnest p2 -> alias to ts2/svc2/msg2  (no level in our syslog template)
    s2 = (
        s1
        .unnest("p2")
        .select(
            "source_path", "lineno", "text", "p3", "tid",
            "ts1", "lvl1", "svc1", "msg1",
            fc.col("timestamp").alias("ts2"),
            fc.col("service").alias("svc2"),
            fc.col("message").alias("msg2"),
        )
    )

    # Unnest p3 -> alias to ts3/lvl3/svc3/msg3
    s3 = (
        s2
        .unnest("p3")
        .select(
            "source_path", "lineno", "text", "tid",
            "ts1", "lvl1", "svc1", "msg1",
            "ts2", "svc2", "msg2",
            fc.col("timestamp").alias("ts3"),
            fc.col("level").alias("lvl3"),
            fc.col("service").alias("svc3"),
            fc.col("message").alias("msg3"),
        )
    )

    # Unnest tid -> alias to trace_id
    s4 = (
        s3
        .unnest("tid")
        .select(
            "source_path", "lineno", "text",
            "ts1", "lvl1", "svc1", "msg1",
            "ts2", "svc2", "msg2",
            "ts3", "lvl3", "svc3", "msg3",
            fc.col("trace_id").alias("trace_id"),
        )
    )

    # Final coalesced columns (ISO → SYSLOG → PYLOG for ts/svc/msg; ISO/PYLOG for level)
    return s4.select(
        "source_path", "lineno", "text",
        fc.coalesce(fc.col("ts1"),  fc.col("ts2"),  fc.col("ts3")).alias("timestamp"),
        fc.coalesce(fc.col("lvl1"),                 fc.col("lvl3")).alias("level"),
        fc.coalesce(fc.col("svc1"), fc.col("svc2"), fc.col("svc3"), fc.lit("unknown")).alias("service"),
        fc.coalesce(fc.col("msg1"), fc.col("msg2"), fc.col("msg3"), fc.col("text")).alias("message"),
        "trace_id",
    )

# Apply
parsed_df = parse_lines(logs_df)
print("Parsed rows:", parsed_df.count())
parsed_df.limit(10).show()

Parsed rows: 10
┌──────────────┬────────┬─────────────┬─────────────┬───────┬─────────────┬─────────────┬──────────┐
│ source_path  ┆ lineno ┆ text        ┆ timestamp   ┆ level ┆ service     ┆ message     ┆ trace_id │
╞══════════════╪════════╪═════════════╪═════════════╪═══════╪═════════════╪═════════════╪══════════╡
│ /content/dat ┆ 1      ┆ 2025-09-20T ┆ 2025-09-20T ┆ INFO  ┆ web         ┆ GET /health ┆ null     │
│ a/logs/app1. ┆        ┆ 10:14:02Z   ┆ 10:14:02Z   ┆       ┆             ┆ 200 in 3ms  ┆          │
│ log          ┆        ┆ [INFO] web: ┆             ┆       ┆             ┆             ┆          │
│              ┆        ┆ GET /health ┆             ┆       ┆             ┆             ┆          │
│              ┆        ┆ 200 in 3ms  ┆             ┆       ┆             ┆             ┆          │
│ /content/dat ┆ 2      ┆ 2025-09-20T ┆ 2025-09-20T ┆ WARN  ┆ db          ┆ Connection  ┆ null     │
│ a/logs/app1. ┆        ┆ 10:14:05Z   ┆ 10:14:05Z   ┆       ┆             ┆

## **Step 4: Stable fingerprinting with `semantic.extract` → `fingerprinted_df`**

**What this cell does**

* Uses **LLM-assisted extraction** (`semantic.extract`) to pull structured parts from each `message`:

  * `symbol` (e.g., `TimeoutError`, `unique_constraint_failed`)

  * `file`, `function` (from stack traces if present)

  * `stem` (a normalized, noise-reduced paraphrase of the message)

* Flattens the struct with **`unnest("sx")`** and builds a compact, comparable fingerprint:

   `[service] | [symbol] | [file#function] | [stem]`



In [ ]:
# --- Step 4: Fingerprinting (noise-tolerant grouping key) ---

from typing import Optional
from pydantic import BaseModel, Field

# Describe every field (Fenic validates these)
class FingerprintOut(BaseModel):
    """Structured fingerprint parts extracted from a log line."""
    symbol:   Optional[str] = Field(
        None,
        description="Error symbol or condition tag (e.g., TimeoutError, unique_constraint_failed)."
    )
    file:     Optional[str] = Field(
        None,
        description="Source file path from a stack trace if present (e.g., /srv/app/handlers/user.py)."
    )
    function: Optional[str] = Field(
        None,
        description="Function name from a stack trace if present (e.g., handle)."
    )
    stem:     str = Field(
        ...,
        description="Normalized message stem with volatile tokens removed; concise summary of the event."
    )

def fingerprint(df):
    # LLM-powered structured extraction -> struct column "sx"
    sx = fc.semantic.extract(
        fc.col("message"),
        response_format=FingerprintOut,  # Pydantic model (with descriptions)
        model_alias="mini",              # must be defined in your Step 1
    ).alias("sx")

    # Add and flatten the struct
    with_ex = df.select(
        "source_path","lineno","timestamp","level","service","message","trace_id",
        sx
    )
    flat = with_ex.unnest("sx")

    # Compose a stable fingerprint
    sym_piece  = fc.coalesce(fc.col("symbol"),   fc.lit("no_symbol"))
    call_file  = fc.coalesce(fc.col("file"),     fc.lit("no_file"))
    call_func  = fc.coalesce(fc.col("function"), fc.lit("no_func"))
    stem_piece = fc.coalesce(fc.col("stem"),     fc.col("message"))

    fp = fc.text.concat(
        fc.coalesce(fc.col("service"), fc.lit("svc:unknown")), fc.lit(" | "),
        sym_piece, fc.lit(" | "),
        call_file, fc.lit("#"), call_func, fc.lit(" | "),
        stem_piece
    ).alias("fingerprint")

    return flat.select(
        "source_path","lineno","timestamp","level","service","message","trace_id",
        "symbol","file","function","stem",
        fp
    )

# Apply
fingerprinted_df = fingerprint(parsed_df)
print("Fingerprint rows:", fingerprinted_df.count())
fingerprinted_df.limit(10).show()

Submitting requests for batch: 7e3e2620-1f4b-407e-b268-1b50241196d1 (model: gpt-4o-mini): 100%|██████████| 10/10 [00:00<00:00, 130.60req/s, estimated_input_tokens=3261, estimated_output_tokens=10240]
Awaiting responses for batch 7e3e2620-1f4b-407e-b268-1b50241196d1 (model: gpt-4o-mini): 100%|██████████| 10/10 [00:02<00:00,  4.86res/s]


Fingerprint rows: 10


Submitting requests for batch: d955f540-4a04-4736-8862-05e7ac374a0f (model: gpt-4o-mini): 100%|██████████| 10/10 [00:00<00:00, 146.53req/s, estimated_input_tokens=3261, estimated_output_tokens=10240]
Awaiting responses for batch d955f540-4a04-4736-8862-05e7ac374a0f (model: gpt-4o-mini): 100%|██████████| 10/10 [00:02<00:00,  3.53res/s]

┌─────────────┬────────┬─────────────┬───────┬───┬────────────┬──────────┬────────────┬────────────┐
│ source_path ┆ lineno ┆ timestamp   ┆ level ┆ … ┆ file       ┆ function ┆ stem       ┆ fingerprin │
│             ┆        ┆             ┆       ┆   ┆            ┆          ┆            ┆ t          │
╞═════════════╪════════╪═════════════╪═══════╪═══╪════════════╪══════════╪════════════╪════════════╡
│ /content/da ┆ 1      ┆ 2025-09-20T ┆ INFO  ┆ … ┆ null       ┆ null     ┆ Successful ┆ web |      │
│ ta/logs/app ┆        ┆ 10:14:02Z   ┆       ┆   ┆            ┆          ┆ health     ┆ no_symbol  │
│ 1.log       ┆        ┆             ┆       ┆   ┆            ┆          ┆ check with ┆ | no_file# │
│             ┆        ┆             ┆       ┆   ┆            ┆          ┆ 200        ┆ no_func |  │
│             ┆        ┆             ┆       ┆   ┆            ┆          ┆ response   ┆ Successful │
│             ┆        ┆             ┆       ┆   ┆            ┆          ┆ in 3ms.    ┆ hea

## **Step 5: Incident severity tagging (rule-first, LLM-optional) → `triage_df`**

**What this cell does**

* Assigns each row a **severity** (`info`, `warn`, `error`) plus a numeric **`severity_score`** (`0.20`, `0.60`, `0.90`) for easy sorting.

* Uses **deterministic string rules** only (no model required). You can layer an LLM pass later if you want.

**How it works (quick logic)**

* Builds lowercase helpers:  
   `msg_lc`, `level_norm`, `svc_lc` (digits preserved so HTTP **5xx** stays detectable).

* Flags common error patterns:

  * **Level** keywords: `error|fatal|crit`

  * **Symptoms**: `exception`, `traceback`, `timeoutError`, `OperationalError`, `connection refused`, `unique constraint`

  * **HTTP**: `5xx` via `\b5\d{2}\b`

  * **Nginx upstream** problems (closed connection / too big header)

* Else if **warn** patterns (`warn`, `retry`, `slow`, `latency`, `degraded`) → `warn`; otherwise `info`.

**Why this approach**

* SREs need **stable, auditable** tagging. Clear rules make it easy to review and tune.

* Keeping digits intact avoids false negatives on status codes and numeric signals.

* The numeric score lets you sort tables without string ordering quirks.  

In [ ]:
# --- Step 5: Severity tagging ---

def tag_severity_additive(df):
    # Reusable lowercase variants
    msg_lc     = fc.text.lower(fc.coalesce(fc.col("message"), fc.lit(""))).alias("msg_lc")
    level_norm = fc.text.lower(fc.coalesce(fc.col("level"),   fc.lit(""))).alias("level_norm")
    svc_lc     = fc.text.lower(fc.coalesce(fc.col("service"), fc.lit(""))).alias("svc_lc")

    base = df.select("*", msg_lc, level_norm, svc_lc)

    # Regex predicates
    http_5xx = fc.col("msg_lc").rlike(r"\b5\d{2}\b")  # 500–599
    nginx_upstream_err = (
        (fc.col("svc_lc").contains("nginx") | fc.col("msg_lc").contains("nginx"))
        & fc.col("msg_lc").rlike(r"(upstream|closed connection|too big header)")
    )

    # Stage 1: compute severity
    severity = (
        fc.when(
            fc.col("level_norm").contains("error")
            | fc.col("level_norm").contains("fatal")
            | fc.col("level_norm").contains("crit")
            | fc.col("msg_lc").contains("exception")
            | fc.col("msg_lc").contains("traceback")
            | fc.col("msg_lc").contains("timeouterror")
            | fc.col("msg_lc").contains("operationalerror")
            | fc.col("msg_lc").contains("connection refused")
            | fc.col("msg_lc").contains("unique constraint")
            | http_5xx
            | nginx_upstream_err,
            fc.lit("error")
        )
        .otherwise(
            fc.when(
                fc.col("level_norm").contains("warn")
                | fc.col("msg_lc").contains("retry")
                | fc.col("msg_lc").contains("slow")
                | fc.col("msg_lc").contains("latency")
                | fc.col("msg_lc").contains("degraded"),
                fc.lit("warn")
            ).otherwise(fc.lit("info"))
        )
    ).alias("severity")

    step1 = base.select("*", severity)

    # Stage 2: compute severity_score (now severity exists)
    sev = fc.col("severity")
    severity_score = (
        fc.when(sev == fc.lit("error"), fc.lit(0.90))
        .otherwise(fc.when(sev == fc.lit("warn"), fc.lit(0.60)).otherwise(fc.lit(0.20)))
    ).alias("severity_score")

    # Keep only original columns + derived outputs (drop helpers)
    return step1.select(*df.columns, sev, severity_score)

# Re-tag from your fingerprinted_df
triage_df = tag_severity_additive(fingerprinted_df)
print("Severity rows:", triage_df.count())
triage_df.limit(10).show()

Submitting requests for batch: 9e4ca86e-f16c-4491-a6bc-9e6080f8e558 (model: gpt-4o-mini): 100%|██████████| 10/10 [00:00<00:00, 23.44req/s, estimated_input_tokens=3261, estimated_output_tokens=10240]
Awaiting responses for batch 9e4ca86e-f16c-4491-a6bc-9e6080f8e558 (model: gpt-4o-mini): 100%|██████████| 10/10 [00:02<00:00,  4.85res/s]


Severity rows: 10


Submitting requests for batch: e78e1549-38ff-4a8c-b825-a312608ee67b (model: gpt-4o-mini): 100%|██████████| 10/10 [00:00<00:00, 122.51req/s, estimated_input_tokens=3261, estimated_output_tokens=10240]
Awaiting responses for batch e78e1549-38ff-4a8c-b825-a312608ee67b (model: gpt-4o-mini): 100%|██████████| 10/10 [00:01<00:00,  6.94res/s]

┌─────────────┬────────┬─────────────┬───────┬───┬────────────┬────────────┬──────────┬────────────┐
│ source_path ┆ lineno ┆ timestamp   ┆ level ┆ … ┆ stem       ┆ fingerprin ┆ severity ┆ severity_s │
│             ┆        ┆             ┆       ┆   ┆            ┆ t          ┆          ┆ core       │
╞═════════════╪════════╪═════════════╪═══════╪═══╪════════════╪════════════╪══════════╪════════════╡
│ /content/da ┆ 1      ┆ 2025-09-20T ┆ INFO  ┆ … ┆ Health     ┆ web |      ┆ info     ┆ 0.2        │
│ ta/logs/app ┆        ┆ 10:14:02Z   ┆       ┆   ┆ check      ┆ no_symbol  ┆          ┆            │
│ 1.log       ┆        ┆             ┆       ┆   ┆ succeeded  ┆ | no_file# ┆          ┆            │
│             ┆        ┆             ┆       ┆   ┆ with       ┆ no_func |  ┆          ┆            │
│             ┆        ┆             ┆       ┆   ┆ status 200 ┆ Health     ┆          ┆            │
│             ┆        ┆             ┆       ┆   ┆            ┆ check      ┆          ┆    

### **Step 6: Severity-aware semantic clustering**

**What it does**

1. **Enrich text** → `sem_text = "[svc:…] [sev:…] …"` to give clustering more context.

2. **Embed once** → `semantic.embed(sem_text)` and **cache** the result.

3. **Cluster per severity** → K-Means via `with_cluster_labels(emb, ...)`.

4. **Score** each row by **distance to centroid**: `score = 1 - cosine(emb, centroid)` (lower \= closer).

5. **Pick exemplars** → per `cluster_id`, take the row with **min(score)** as the representative message.

6. **Produce two views**:

   * `clusters_df`: one row per cluster with `cluster_id, severity, fingerprint, count, sample_lines, summary`.

   * `assignments_df`: one row per original log with `cluster_id, score, evidence`.




In [ ]:
# --- Step 6: Fenic-native K-Means ---

import math

# 1) Build enriched text for embedding to improve clustering signal: [svc] [sev] message
sem_text = fc.text.concat(
    fc.lit("[svc:"), fc.coalesce(fc.col("service"), fc.lit("unknown")), fc.lit("] "),
    fc.lit("[sev:"), fc.coalesce(fc.col("severity"), fc.lit("info")),   fc.lit("] "),
    fc.col("message")
).alias("sem_text")

with_semtext = triage_df.select(
    "source_path","lineno","timestamp","level","service","message","trace_id",
    "severity","severity_score","fingerprint",
    sem_text
)

# 2) Embed the enriched text (uses the embed model alias registered in Step 1)
with_emb = with_semtext.select(
    "*",
    fc.semantic.embed(fc.col("sem_text"), model_alias="embed").alias("emb")
).cache()

# 3) Helper: cluster a single severity bin and compute a distance score with Fenic
def _cluster_bin(df_sev, offset):
    n = df_sev.count()
    if n == 0:
        return None, offset

    # K heuristic per severity bin: at least 2 if grp>=2; else 1; max 7
    k = 1 if n == 1 else min(7, max(2, int(round(math.sqrt(n)))))

    clustered = (
        df_sev.semantic.with_cluster_labels(
            "emb",
            num_clusters=k,
            num_init=8,
            max_iter=100,
            label_column="local_label",
            centroid_column="centroid"
        )
        .select(
            "*",
            (fc.col("local_label") + fc.lit(offset)).alias("cluster_id"),
            # distance-to-centroid (lower = closer/better exemplar)
            (fc.lit(1.0) - fc.embedding.compute_similarity(fc.col("emb"), fc.col("centroid"), metric="cosine")).alias("score")
        )
    )
    return clustered, offset + k

# 4) Cluster separately per severity, then union results
ordered_bins = ["info", "warn", "error"]
offset = 0
unioned = None

for sev in ordered_bins:
    cbin, offset = _cluster_bin(with_emb.filter(fc.col("severity") == fc.lit(sev)), offset)
    if cbin is None:
        continue
    unioned = cbin if unioned is None else unioned.union(cbin.select(*unioned.columns))

# 5) Build final views (clusters + assignments)
if unioned is None:
    clusters_df = session.create_dataframe(
        {"cluster_id": [], "severity": [], "fingerprint": [], "summary": [], "count": [], "sample_lines": []}
    )
    assignments_df = session.create_dataframe(
        {"source_path": [], "lineno": [], "cluster_id": [], "score": [], "evidence": []}
    )
else:
    # Assignments (one row per original log)
    assignments_df = unioned.select(
        "source_path","lineno","cluster_id","score",
        fc.col("fingerprint").alias("evidence")
    )

    # Exemplar per cluster = row with min(score)
    mins = unioned.group_by("cluster_id").agg(fc.min(fc.col("score")).alias("min_score"))
    exemplars = (
        unioned.join(mins, on="cluster_id", how="inner")
               .filter(fc.col("score") == fc.col("min_score"))
               .select("cluster_id", "severity", "fingerprint", fc.col("message").alias("sample_lines"))
    )

    counts = unioned.group_by("cluster_id").agg(fc.count("*").alias("count"))

    clusters_df = (
        exemplars.join(counts, on="cluster_id", how="inner")
                 .select(
                     "cluster_id",
                     "severity",
                     "fingerprint",
                     fc.text.concat(
                         fc.text.upper(fc.col("severity")), fc.lit(" · "),
                         fc.col("count"),                  fc.lit(" events · e.g. "),
                         fc.col("sample_lines")
                     ).alias("summary"),
                     "count",
                     "sample_lines"
                 )
                 .order_by("count", ascending=False)
    )

print("Clusters:")
clusters_df.limit(10).show()

print("\nSample assignments:")
assignments_df.limit(12).show()

Submitting requests for batch: 7901530d-7dba-4017-96ad-1e9b0b2d38a2 (model: gpt-4o-mini): 100%|██████████| 10/10 [00:00<00:00, 89.80req/s, estimated_input_tokens=3261, estimated_output_tokens=10240]
Awaiting responses for batch 7901530d-7dba-4017-96ad-1e9b0b2d38a2 (model: gpt-4o-mini): 100%|██████████| 10/10 [00:03<00:00,  3.21res/s]
Submitting requests for batch: c72f39e0-dc75-47ae-9ddf-62b54a84f6a7 (model: text-embedding-3-small): 100%|██████████| 10/10 [00:00<00:00, 127.33req/s, estimated_input_tokens=391, estimated_output_tokens=0]
Awaiting responses for batch c72f39e0-dc75-47ae-9ddf-62b54a84f6a7 (model: text-embedding-3-small): 100%|██████████| 10/10 [00:01<00:00,  8.39res/s]


Clusters:
┌────────────┬──────────┬──────────────────────┬─────────────────────┬───────┬─────────────────────┐
│ cluster_id ┆ severity ┆ fingerprint          ┆ summary             ┆ count ┆ sample_lines        │
╞════════════╪══════════╪══════════════════════╪═════════════════════╪═══════╪═════════════════════╡
│ 5          ┆ error    ┆ api | TimeoutError | ┆ ERROR · 3 events ·  ┆ 3     ┆ TimeoutError on     │
│            ┆          ┆ /srv/app/handlers/us ┆ e.g. TimeoutError   ┆       ┆ /v1/users after     │
│            ┆          ┆ er.py#handle |       ┆ on /v1/users after  ┆       ┆ 30010ms             │
│            ┆          ┆ Timeout error        ┆ 30010ms             ┆       ┆ trace_id=abc-123    │
│            ┆          ┆ occurred after       ┆ trace_id=abc-123    ┆       ┆ Traceback (most     │
│            ┆          ┆ 30010ms on           ┆ Traceback (most     ┆       ┆ recent call last):  │
│            ┆          ┆ /v1/users.           ┆ recent call last):  ┆       ┆ Fi

## **Step 7: Metrics, artifacts, and a summary**



**What this step does**

* **Compute coverage**: how many unique log lines we produced assignments for vs. total parsed lines — entirely in **Fenic** (no pandas/Polars).

* **Score clusters**: `score = severity_weight × count` where `ERROR=5`, `WARN=3`, `INFO=1`, then sort to surface the **top cluster**.

* **Export artifacts**: write `clusters.csv`, `assignments.csv`, `clusters.json`, `assignments.json`, and a human-readable `report.md` using only Python’s stdlib.

* **Produce a log error summary** string that you can post from any workflow (LangGraph, cron, CI).


In [ ]:
# --- Step 7: Metrics, artifacts, and error log summary ---

import csv, json

OUT_DIR = Path("/content/out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- 1) Coverage ----------------
def _distinct_pairs(df):
    # Unique (source_path, lineno) rows = number of groups
    return df.group_by("source_path", "lineno").agg(fc.count(fc.lit(1)).alias("c")).count()

# Optional guard to make ordering explicit (nice error if user skips steps)
assert "triage_df" in globals(), "Run Step 5 first to create triage_df"
assert "assignments_df" in globals(), "Run Step 6 first to create assignments_df"

total_lines = _distinct_pairs(triage_df)        # all parsed/triaged lines
processed_lines = _distinct_pairs(assignments_df)  # lines assigned to clusters
coverage = processed_lines / max(1, total_lines)

# --------------- 2) Score + top cluster ---------------
# Sev weight: ERROR=5, WARN=3, INFO=1
sev_weight = (
    fc.when(fc.col("severity") == fc.lit("error"), fc.lit(5))
      .otherwise(fc.when(fc.col("severity") == fc.lit("warn"), fc.lit(3)).otherwise(fc.lit(1)))
)

scored = (
    clusters_df
    .select(
        "cluster_id", "severity", "fingerprint", "summary", "count",
        (fc.col("count") * sev_weight).alias("score")
    )
    .order_by(["score", "count"], ascending=[False, False])
)

top1 = scored.limit(1)

# --------------- 3) Tiny stdlib writers ---------------
def _rows(df):
    """Fenic DF -> list[dict] via to_pydict()."""
    cols = df.columns
    data = df.to_pydict()  # {col: [vals...]}
    return [dict(zip(cols, vals)) for vals in zip(*[data[c] for c in cols])]

def _write_csv(path, rows, cols):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for r in rows:
            w.writerow({c: ("" if r.get(c) is None else str(r.get(c))) for c in cols})

def _md_table(rows, cols):
    if not rows:
        return "_No rows._"
    head = "| " + " | ".join(cols) + " |\n"
    sep  = "| " + " | ".join(["---"] * len(cols)) + " |\n"
    body = "\n".join("| " + " | ".join(str(r.get(c, ""))[:200] for c in cols) + " |" for r in rows)
    return head + sep + body

# --------------- 4) Artifacts (CSV/JSON/Markdown) ---------------
cl_rows = _rows(scored)
as_rows = _rows(assignments_df.select("source_path","lineno","cluster_id","score", fc.col("evidence")))

# CSV
cl_csv = OUT_DIR / "clusters.csv"
as_csv = OUT_DIR / "assignments.csv"
_write_csv(cl_csv, cl_rows, ["cluster_id","severity","count","score","fingerprint","summary"])
_write_csv(as_csv, as_rows, ["source_path","lineno","cluster_id","score","evidence"])

# JSON
(cl_json := OUT_DIR / "clusters.json").write_text(json.dumps(cl_rows, indent=2), encoding="utf-8")
(as_json := OUT_DIR / "assignments.json").write_text(json.dumps(as_rows, indent=2), encoding="utf-8")

# Markdown
top_list = _rows(top1)
md = [
    "# Fenic — Daily Log Triage (Demo)",
    "",
    f"- **Coverage**: {processed_lines}/{total_lines} = **{coverage:.2%}**",
    f"- **Clusters**: {len(cl_rows)}",
]
if top_list:
    t = top_list[0]
    md.append(f"- **Top cluster**: #{int(t['cluster_id'])} · **{str(t['severity']).upper()}** · {int(t['count'])} events")
md += [
    "",
    "## Clusters",
    _md_table(sorted(cl_rows, key=lambda r: (r['score'], r['count']), reverse=True),
              ["cluster_id","severity","count","fingerprint","summary"]),
    "",
    "## Sample assignments",
    "(first 12 rows)",
    _md_table(as_rows[:12], ["source_path","lineno","cluster_id","score","evidence"])
]
(OUT_DIR / "report.md").write_text("\n".join(md), encoding="utf-8")

# --------------- 5) Error log summary ---------------
if top_list:
    t = top_list[0]
    slack_md = (
        f"*Daily log triage* — coverage {coverage:.0%}\n"
        f"• *Top cluster*: `#{int(t['cluster_id'])}` · *{str(t['severity']).upper()}* · {int(t['count'])} events\n"
        f"• Summary: {str(t['summary'])}\n"
        f"• Fingerprint: `{str(t['fingerprint'])[:140]}`\n"
        f"_See clusters.csv / report.md for details._"
    )
else:
    slack_md = f"*Daily log triage* — coverage {coverage:.0%}\n_No actionable clusters today._"

print("Artifacts written:")
for p in (cl_csv, as_csv, cl_json, as_json, OUT_DIR / "report.md"):
    print(" -", str(p))
print("\nSlack summary:\n", slack_md)


Submitting requests for batch: 57c0efe2-8a1e-46b2-bf5a-e3c4de0fc86b (model: gpt-4o-mini): 100%|██████████| 10/10 [00:00<00:00, 68.33req/s, estimated_input_tokens=3261, estimated_output_tokens=10240]
Awaiting responses for batch 57c0efe2-8a1e-46b2-bf5a-e3c4de0fc86b (model: gpt-4o-mini): 100%|██████████| 10/10 [00:02<00:00,  4.72res/s]


Artifacts written:
 - /content/out/clusters.csv
 - /content/out/assignments.csv
 - /content/out/clusters.json
 - /content/out/assignments.json
 - /content/out/report.md

Slack summary:
 *Daily log triage* — coverage 100%
• *Top cluster*: `#4` · *ERROR* · 3 events
• Summary: ERROR · 3 events · e.g. 502 upstream prematurely closed connection while reading response header
• Fingerprint: `10:14:01 edge-1 nginx | no_symbol | no_file#no_func | 502 upstream prematurely closed connection while reading response header`
_See clusters.csv / report.md for details._


## **Expose Fenic as MCP Tools (Catalog → Server)**


Turn your Fenic queries into **callable tools** and serve them over an MCP endpoint that LangGraph (or any MCP client) can hit.


**What’s inside**

* Three read tools:

  * `list_clusters(severity_floor="info")` — severity-weighted ranking.

  * `clusters_by_severity(severity)` — slice clusters by a single severity.

  * `assignments_for_cluster(cluster_id)` — raw evidence lines for one cluster.

* One metrics tool:

  * `coverage_metrics()` — `processed_lines`, `total_lines`, `coverage_ratio`.

* All tools are **pure Fenic DataFrame plans**; no external conversions.

* A small server bootstrap that **starts MCP in the background** (so your notebook remains usable).

**Requirements checklist**

* You ran Steps 1–6 and have these DataFrames: `triage_df`, `clusters_df`, `assignments_df`.

* Install the MCP extra:

  * `pip install "fenic[mcp]" fastmcp`

* Environment vars for embed models (e.g., `OPENAI_API_KEY`) are set if your tools depend on them.


**How responses look**

* MCP returns a **tabular payload** that Fenic serializes: a Markdown table plus simple metadata (row count, schema).

* In LangGraph or a client, you’ll usually print the **Markdown** (human-friendly) or parse rows if you want programmatic routing.

**What to expect**

* `list_clusters("warn")` returns a table with `cluster_id | severity | count | fingerprint | summary | sev_w | score` sorted by importance.

* `coverage_metrics()` returns a **single row** with `processed_lines`, `total_lines`, `coverage_ratio`.

* The server logs each tool call with **execution IDs** and timing.


**Security & scope**

* This demo serves on **localhost**. For real deployments, run behind an API gateway and configure auth.

* Tools only expose **read** queries registered in your catalog.

**Smoke test (high level)**

1. Start the MCP server cell; look for “**MCP server started in background** …”.

2. Probe with a tiny client (or `httpx.post`) and call:

   * `tools/list`

   * `tools/call` → `list_clusters` with `{"params":{"severity_floor":"warn"}}`

3. You should see the Markdown table of clusters and a 1-row coverage table.

**Clean up**

* If you need to stop the server, cancel the background task variable you created (`mcp_task.cancel()` in an awaited cell), or restart the kernel.

In [ ]:
# --- MCP appendix ---

import asyncio, socket
import fenic as fc
from fenic.core.mcp.types import ToolParam
from fenic.api.mcp.server import create_mcp_server, run_mcp_server_async

# Prefer fc.tool_param; fall back for older versions.
try:
    tool_param = fc.tool_param
except AttributeError:
    from fenic.core._logical_plan.expressions.basic import tool_param  # compat

# 0) Persist results as catalog tables (idempotent)
triage_df.write.save_as_table("triage", mode="overwrite")
clusters_df.write.save_as_table("clusters", mode="overwrite")
assignments_df.write.save_as_table("assignments", mode="overwrite")

triage      = session.table("triage")
clusters    = session.table("clusters")
assignments = session.table("assignments")

# 1) list_clusters(severity_floor="info")
sev = fc.col("severity")
sev_w = (
    fc.when(sev == fc.lit("error"), fc.lit(5))
      .otherwise(fc.when(sev == fc.lit("warn"), fc.lit(3)).otherwise(fc.lit(1)))
).alias("sev_w")

severity_floor = tool_param("severity_floor", fc.StringType)

allowed = (
    fc.when(severity_floor == fc.lit("error"), sev == fc.lit("error"))
      .otherwise(
          fc.when(severity_floor == fc.lit("warn"),
                  (sev == fc.lit("warn")) | (sev == fc.lit("error")))
            .otherwise(fc.lit(True))
      )
)

cluster_listing = (
    clusters
      .filter(allowed)
      .select(
          "cluster_id", "severity", "count", "fingerprint", "summary",
          sev_w,
          (fc.col("count") * sev_w).alias("score"),
      )
      .order_by(["score", "count"], ascending=[False, False])
)

session.catalog.create_tool(
    tool_name="list_clusters",
    tool_description="List clusters ordered by severity-weighted importance. severity_floor in {info,warn,error}.",
    tool_query=cluster_listing,
    tool_params=[
        ToolParam(
            name="severity_floor",
            description="Minimum severity to include (info|warn|error)",
            has_default=True,
            default_value="info",
        )
    ],
)

# 2) clusters_by_severity(severity)
severity_param = tool_param("severity", fc.StringType)
clusters_by_sev = clusters.filter(sev == severity_param).select(
    "cluster_id","severity","count","fingerprint","summary"
)

session.catalog.create_tool(
    tool_name="clusters_by_severity",
    tool_description="Return clusters for a single severity.",
    tool_query=clusters_by_sev,
    tool_params=[ToolParam(name="severity", description="One of: info | warn | error")],
)

# 3) assignments_for_cluster(cluster_id)
cluster_id_param = tool_param("cluster_id", fc.IntegerType)
assignments_for_cluster = assignments.filter(
    fc.col("cluster_id") == cluster_id_param
).select("source_path","lineno","cluster_id","score","evidence")

session.catalog.create_tool(
    tool_name="assignments_for_cluster",
    tool_description="List assignments (raw lines) for a specific cluster_id.",
    tool_query=assignments_for_cluster,
    tool_params=[ToolParam(name="cluster_id", description="Cluster id (integer)")],
    result_limit=1000,
)

# 4) coverage_metrics()
tri_total = (
    triage.group_by("source_path","lineno")
          .agg(fc.count(fc.lit(1)).alias("n"))
          .select(fc.lit(1).alias("k"))
          .group_by("k").agg(fc.count(fc.lit(1)).alias("total_lines"))
)

as_total = (
    assignments.group_by("source_path","lineno")
               .agg(fc.count(fc.lit(1)).alias("n"))
               .select(fc.lit(1).alias("k"))
               .group_by("k").agg(fc.count(fc.lit(1)).alias("processed_lines"))
)

metrics_df = (
    tri_total.join(as_total, on="k", how="left")
             .select(
                 fc.col("processed_lines").alias("processed_lines"),
                 fc.col("total_lines").alias("total_lines"),
                 fc.text.concat(fc.col("processed_lines"), fc.lit("/"), fc.col("total_lines")).alias("coverage_ratio"),
             )
)

session.catalog.create_tool(
    tool_name="coverage_metrics",
    tool_description="Basic coverage metrics (processed_lines, total_lines, coverage_ratio).",
    tool_query=metrics_df,
    tool_params=[],          # explicit: no parameters
    result_limit=1,          # it returns a single summary row
)

# 5) Serve tools
tools = session.catalog.list_tools()
server = create_mcp_server(session, server_name="Fenic MCP (demo)", tools=tools)

def find_free_port(host="127.0.0.1"):
    import socket as _socket
    s = _socket.socket(_socket.AF_INET, _socket.SOCK_STREAM)
    s.bind((host, 0))
    _, p = s.getsockname()
    s.close()
    return p

HOST, PORT = "127.0.0.1", find_free_port()

# Start server in background (don't await)
mcp_task = asyncio.create_task(
    run_mcp_server_async(
        server,
        transport="http",
        host=HOST,
        port=PORT,
        stateless_http=True,
        path="/mcp",
        log_level="warning",
    )
)

# tiny delay lets uvicorn bind the socket
await asyncio.sleep(0.5)
print(f"MCP HTTP server running at http://{HOST}:{PORT}/mcp")

### Smoke test the MCP server

In [ ]:
from fastmcp import Client
HOST, PORT = "127.0.0.1", PORT

def show_table(result, title=None, max_chars=2000):
    d = getattr(result, "data", result)
    rows = (d.rows or "").replace("\r", "").replace("\n\n", "\n")
    if title: print(f"\n=== {title} ===")
    print(rows[:max_chars])

async def probe():
    async with Client(f"http://{HOST}:{PORT}/mcp") as c:
        tools = await c.list_tools()
        print("Tools:", [t.name for t in tools])

        res = await c.call_tool("list_clusters", arguments={"params": {"severity_floor": "warn"}})
        show_table(res, "list_clusters(warn)")

        res2 = await c.call_tool("coverage_metrics", arguments={"params": {}})
        show_table(res2, "coverage_metrics")

        res3 = await c.call_tool("clusters_by_severity", arguments={"params": {"severity": "error"}})
        show_table(res3, "clusters_by_severity(error)")

        res4 = await c.call_tool("assignments_for_cluster", arguments={"params": {"cluster_id": 0}})
        show_table(res4, "assignments_for_cluster(0)")

# In notebooks with a running loop:
task = asyncio.create_task(probe())
await task

INFO:fenic._backends.local.execution:Execution ID: ab09d4f1-144e-4071-b370-4c4547c519c7
INFO:fenic._backends.local.system_table_client:Appended metrics for execution ab09d4f1-144e-4071-b370-4c4547c519c7
INFO:fenic.core.mcp._server:Completed query for list_clusters in 17ms with 4 result rows.


Tools: ['list_clusters', 'clusters_by_severity', 'assignments_for_cluster', 'coverage_metrics']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO:fenic._backends.local.execution:Execution ID: 010ed712-e5e3-464a-901f-1afa715a2d47
INFO:fenic._backends.local.system_table_client:Appended metrics for execution 010ed712-e5e3-464a-901f-1afa715a2d47
INFO:fenic.core.mcp._server:Completed query for coverage_metrics in 47ms with 1 result rows.



=== list_clusters(warn) ===
| cluster_id | severity | count | fingerprint | summary | sev_w | score |
| --- | --- | --- | --- | --- | --- | --- |
| 4 | error | 3 | 10:14:01 edge-1 nginx | no_symbol | no_file#no_func | 502 upstream prematurely closed connection while reading response header | ERROR · 3 events · e.g. 502 upstream prematurely closed connection while reading response header | 5 | 15 |
| 5 | error | 3 | api | TimeoutError | /srv/app/handlers/user.py#handle | Timeout error occurred after 30010ms on /v1/users. | ERROR · 3 events · e.g. TimeoutError on /v1/users after 30010ms trace_id=abc-123
Traceback (most recent call last):
  File "/srv/app/handlers/user.py", line 214, in handle
    resp = client.get(url, timeout=30)
  File "/usr/local/lib/python3.11/site-packages/requests/api.py", line 73, in get
    return request('get', url, params=params, **kwargs)
requests.exceptions.Timeout: The request timed out | 5 | 15 |
| 3 | warn | 1 | payment-api | no_symbol | no_file#no_func |

INFO:fenic._backends.local.execution:Execution ID: f506a5fc-b422-40d0-97e0-b4acb5ef2930
INFO:fenic._backends.local.system_table_client:Appended metrics for execution f506a5fc-b422-40d0-97e0-b4acb5ef2930
INFO:fenic.core.mcp._server:Completed query for clusters_by_severity in 16ms with 2 result rows.



=== coverage_metrics ===
| processed_lines | total_lines | coverage_ratio |
| --- | --- | --- |
| 1 | 1 | 1/1 |


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO:fenic._backends.local.execution:Execution ID: 922a326c-685d-4aaa-b5b5-c63dca478fee
INFO:fenic._backends.local.system_table_client:Appended metrics for execution 922a326c-685d-4aaa-b5b5-c63dca478fee
INFO:fenic.core.mcp._server:Completed query for assignments_for_cluster in 19ms with 1 result rows.



=== clusters_by_severity(error) ===
| cluster_id | severity | count | fingerprint | summary |
| --- | --- | --- | --- | --- |
| 4 | error | 3 | 10:14:01 edge-1 nginx | no_symbol | no_file#no_func | 502 upstream prematurely closed connection while reading response header | ERROR · 3 events · e.g. 502 upstream prematurely closed connection while reading response header |
| 5 | error | 3 | api | TimeoutError | /srv/app/handlers/user.py#handle | Timeout error occurred after 30010ms on /v1/users. | ERROR · 3 events · e.g. TimeoutError on /v1/users after 30010ms trace_id=abc-123
Traceback (most recent call last):
  File "/srv/app/handlers/user.py", line 214, in handle
    resp = client.get(url, timeout=30)
  File "/usr/local/lib/python3.11/site-packages/requests/api.py", line 73, in get
    return request('get', url, params=params, **kwargs)
requests.exceptions.Timeout: The request timed out |

=== assignments_for_cluster(0) ===
| source_path | lineno | cluster_id | score | evidence |
| ---

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## **Run the LangGraph agent over your MCP tools**

**What this cell does**

* Sends a natural-language question to a small LangGraph **ReAct** agent.

* The agent calls your **Fenic MCP** tools (`list_clusters`, `clusters_by_severity`, `assignments_for_cluster`, `coverage_metrics`) and returns a Markdown table.

**How to run**

1. Make sure the **MCP server cell is running** (you should see `MCP HTTP server running at http://127.0.0.1:PORT/mcp`).

2. Ensure `OPENAI_API_KEY` is set in the environment.

3. Call the agent with a question, e.g.:

   * `await ask("show top clusters at or above warn")`

   * `await ask("only error clusters")`

   * `await ask("what is our coverage?")`

   * `await ask("show assignments for cluster 5")`

**What to expect**

* A **Markdown table** printed in the output.

* For “top clusters”, you’ll see \~4 rows (two `error`, two `warn`) ordered by severity-weighted score.

* For “coverage”, a 1-row table with `processed_lines`, `total_lines`, `coverage_ratio`.

* For “assignments”, a table of raw lines mapped to a cluster id.

**Troubleshooting**

* If you see timeouts or “no tools available”, re-run the MCP server cell and confirm `HOST`/`PORT` in this agent cell match.

* If you see auth errors, verify `OPENAI_API_KEY` is exported in the kernel.

In [ ]:
# --- LangGraph tool node that calls the Fenic MCP endpoint and run a natural-language LangGraph agent over your MCP tools ---

# deps: pip install langgraph fastmcp langchain-openai

import asyncio, atexit
from typing import Any, Dict, Optional

from fastmcp import Client as MCPClient
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

MCP_URL = f"http://{HOST}:{PORT}/mcp"  # keep in sync with your server cell

# 1) One shared MCP client (single connection)
_mcp_client: Optional[MCPClient] = None

async def _ensure_mcp():
    global _mcp_client
    if _mcp_client is None:
        _mcp_client = MCPClient(MCP_URL)
        await _mcp_client.__aenter__()

@atexit.register
def _shutdown_mcp():
    # sync cleanup (ok if already closed)
    try:
        loop = asyncio.get_event_loop()
        if _mcp_client is not None and loop.is_running():
            loop.create_task(_mcp_client.__aexit__(None, None, None))
    except Exception:
        pass

async def _call_mcp(tool_name: str, params: Optional[Dict[str, Any]] = None):
    await _ensure_mcp()
    # Fenic MCP expects {"params": {...}} even when empty
    res = await _mcp_client.call_tool(tool_name, arguments={"params": params or {}}, raise_on_error=True)
    data = getattr(res, "data", None)
    # Return the human-facing markdown table if present; otherwise a compact fallback
    return (getattr(data, "rows", None)
            or f"(no rows from {tool_name})")

# 2) Expose MCP methods as LangChain tools (no extra wrapper classes)

@tool("list_clusters", return_direct=True)
async def list_clusters(severity_floor: str = "warn") -> str:
    """List clusters ordered by importance. severity_floor in {info|warn|error}."""
    return await _call_mcp("list_clusters", {"severity_floor": severity_floor})

@tool("clusters_by_severity", return_direct=True)
async def clusters_by_severity(severity: str) -> str:
    """Return clusters for a single severity: info | warn | error."""
    return await _call_mcp("clusters_by_severity", {"severity": severity})

@tool("assignments_for_cluster", return_direct=True)
async def assignments_for_cluster(cluster_id: int) -> str:
    """List raw line assignments for a cluster_id."""
    return await _call_mcp("assignments_for_cluster", {"cluster_id": int(cluster_id)})

@tool("coverage_metrics", return_direct=True)
async def coverage_metrics() -> str:
    """Basic coverage metrics (processed_lines / total_lines)."""
    return await _call_mcp("coverage_metrics", {})

tools = [list_clusters, clusters_by_severity, assignments_for_cluster, coverage_metrics]

# 3) LangGraph ReAct agent that chooses among your MCP tools
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
system_prompt = (
    "You are a helpful SRE copilot.\n"
    "Prefer calling the provided tools to answer questions about clusters, coverage, "
    "and raw assignments.\n"
    "For 'top/important' clusters call list_clusters (default floor=warn). "
    "For error-only use clusters_by_severity('error'). "
    "For raw lines use assignments_for_cluster(cluster_id). "
    "For progress use coverage_metrics."
)
agent = create_react_agent(llm, tools, prompt=system_prompt)

# 4) Helper: ask natural-language questions
async def app(q: str) -> str:
    result = await agent.ainvoke({"messages": [{"role": "user", "content": q}]})
    msgs = result["messages"]
    content = msgs[-1].content
    if isinstance(content, list):
        return "\n".join(part.get("text", "") for part in content if isinstance(part, dict) and "text" in part).strip() or str(content)
    return content if isinstance(content, str) else str(content)

print("Agent ready. Try:")
print("- await app('show top clusters at or above warn')")
print("- await app('only error clusters')")
print("- await app('what is our coverage?')")
print("- await app('show assignments for cluster 5')")

Agent ready. Try:
- await app('show top clusters at or above warn')
- await app('only error clusters')
- await app('what is our coverage?')
- await app('show assignments for cluster 5')


/usr/local/lib/python3.12/dist-packages/langchain_core/tools/base.py:1326: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  fields = getattr(cls, "model_fields", {}) or getattr(cls, "__fields__", {})
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from IPython.display import Markdown, display

async def ask(q: str):
    ans = await app(q)
    # If it looks like a Markdown table, render it; otherwise print raw
    if isinstance(ans, str) and ans.lstrip().startswith("|"):
        display(Markdown(ans))
    else:
        print(ans)
    # Return nothing so Jupyter doesn't render the repr
    return

In [ ]:
await ask('show top clusters at or above warn')

INFO:fenic._backends.local.execution:Execution ID: 2737a890-982e-4078-8e0d-be15aa47865f
INFO:fenic._backends.local.system_table_client:Appended metrics for execution 2737a890-982e-4078-8e0d-be15aa47865f
INFO:fenic.core.mcp._server:Completed query for list_clusters in 6ms with 4 result rows.


| cluster_id | severity | count | fingerprint | summary | sev_w | score |
| --- | --- | --- | --- | --- | --- | --- |
| 5 | error | 3 | api | TimeoutError | /srv/app/handlers/user.py#handle | TimeoutError on /v1/users after 30010ms | ERROR · 3 events · e.g. TimeoutError on /v1/users after 30010ms trace_id=abc-123
Traceback (most recent call last):
  File "/srv/app/handlers/user.py", line 214, in handle
    resp = client.get(url, timeout=30)
  File "/usr/local/lib/python3.11/site-packages/requests/api.py", line 73, in get
    return request('get', url, params=params, **kwargs)
requests.exceptions.Timeout: The request timed out | 5 | 15 |
| 4 | error | 3 | 10:14:01 edge-1 nginx | no_symbol | no_file#no_func | Upstream prematurely closed connection while reading response header. | ERROR · 3 events · e.g. 502 upstream prematurely closed connection while reading response header | 5 | 15 |
| 2 | warn | 1 | db | no_symbol | no_file#no_func | Connection slow for shard=3 with latency of 412ms | WARN · 1 events · e.g. Connection slow for shard=3 latency=412ms | 3 | 3 |
| 3 | warn | 1 | payment-api | no_symbol | no_file#no_func | Retrying operation with a delay of 2 seconds (attempt 2 out of 5) | WARN · 1 events · e.g. retrying in 2s (attempt 2/5) | 3 | 3 |

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## **Next Steps**




* Switch normalization to **JSON rows** if you want downstream filters, joins, or ranking inside the graph.


In [ ]:
mcp_task.cancel()

True